[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CharlesShang/TorchCode/blob/master/templates/34_speculative_decoding.ipynb)

# 🔴 Hard: Speculative Decoding

Implement the **acceptance/rejection step** of speculative decoding — a technique for accelerating LLM inference.

### Signature
```python
def speculative_decode(target_probs, draft_probs, draft_tokens) -> list[int]:
    # target_probs: (K, V) from target (large) model
    # draft_probs: (K, V) from draft (small) model
    # draft_tokens: (K,) tokens sampled by draft model
    # Returns: list of accepted tokens (1 to K)
```

### Algorithm
For each position i = 0, ..., K-1:
1. `ratio = target_probs[i, token_i] / draft_probs[i, token_i]`
2. Accept with probability `min(1, ratio)`
3. If rejected: sample from `normalize(max(0, target - draft))`, append, and stop

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.8 MB/s eta 0:00:00


In [2]:
import torch

In [11]:
# ✏️ YOUR IMPLEMENTATION HERE

def speculative_decode(target_probs, draft_probs, draft_tokens):
    accepted = []
    K, V = draft_probs.shape
    for i, token in enumerate(draft_tokens):
        p = target_probs[i, token]
        q = draft_probs[i, token]
        ratio = min(1, p / (q + 1e-8))
        if torch.rand((), device=draft_probs.device) < ratio:
            accepted.append(token.item())
        else:
            adjusted = torch.clamp(target_probs[i] - draft_probs[i], min=0)
            s = adjusted.sum()
            if s > 0:
                  adjusted = adjusted / s
            else:
                  adjusted = torch.ones_like(adjusted) / adjusted.shape[0]
            accepted.append(torch.multinomial(adjusted, 1).item())
    return accepted


In [12]:
# 🧪 Debug
torch.manual_seed(0)
probs = torch.softmax(torch.randn(4, 10), dim=-1)
tokens = torch.tensor([2, 5, 1, 8])
print('Perfect draft:', speculative_decode(probs, probs, tokens))
target = torch.softmax(torch.randn(4, 10), dim=-1)
draft = torch.softmax(torch.randn(4, 10), dim=-1)
print('Random draft:', speculative_decode(target, draft, tokens))

Perfect draft: [2, 5, 1, 8]
Random draft: [2, 5, 1, 8]


In [13]:
# ✅ SUBMIT
from torch_judge import check
check('speculative_decoding')


🧪 Testing: Speculative Decoding (Hard)
──────────────────────────────────────────────────
  ✅ [1/3] Perfect draft: all accepted (4.2ms)
  ✅ [2/3] Output length bounded (2.9ms)
  ✅ [3/3] All tokens valid (22.9ms)
──────────────────────────────────────────────────
  🎉 All 3 tests passed! (29.9ms total)
  Progress saved. Run status() to see your dashboard.

